 1.Importing the Dependencies

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Date/time handling
import datetime as dt

# Ignore warnings (optional)
import warnings
warnings.filterwarnings("ignore")


2. Data Loading and Understanding

In [ ]:

import pandas as pd

# Chicago Crime dataset - pull from City of Chicago API
# Filtering: crimes from 2019 onwards (~5 years of data)
url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$where=date>'2019-01-01T00:00:00'&$limit=500000"

# Read into pandas
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()



In [ ]:
# Convert 'date' column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract features
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

# Quick check
df[['date', 'hour', 'day_of_week', 'month', 'year']].head()


Missing Values Overview

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)


Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=df, y='primary_type', order=df['primary_type'].value_counts().head(10).index)
plt.title("Top 10 Most Common Crime Types (2019–2025)")
plt.xlabel("Count")
plt.ylabel("Crime Type")
plt.show()


Define a “Hotspot”

In [ ]:
# Count crimes by community area
crime_counts = df['community_area'].value_counts()

# Define hotspots as top 10% areas
threshold = crime_counts.quantile(0.90)
hotspots = crime_counts[crime_counts >= threshold].index

# Create hotspot label (1 = hotspot, 0 = not)
df['hotspot'] = df['community_area'].apply(lambda x: 1 if x in hotspots else 0)

df[['community_area','hotspot']].head()


In [ ]:
# Aggregate daily crime counts
daily_counts = df.groupby('date').size().reset_index(name='crime_count')
daily_counts.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(daily_counts['date'], daily_counts['crime_count'])
plt.title("Daily Crime Counts in Chicago (2019–Present)")
plt.xlabel("Date")
plt.ylabel("Number of Crimes")
plt.show()


In [ ]:
df['week'] = pd.to_datetime(df['date']).dt.to_period('W')
weekly_counts = df.groupby('week').size().reset_index(name='crime_count')

plt.figure(figsize=(12,6))
plt.plot(weekly_counts['week'].astype(str), weekly_counts['crime_count'])
plt.title("Weekly Crime Trends")
plt.xlabel("Week")
plt.ylabel("Crimes")
plt.xticks(rotation=45)
plt.show()


Overview of the Data

In [ ]:
# Basic info
df.info()

# Summary statistics
df.describe()

# Check for missing values
df.isnull().sum().sort_values(ascending=False)


Distribution of Crimes Over Time

In [ ]:
# Crimes per year
df['year'] = pd.to_datetime(df['date']).dt.year
yearly_counts = df['year'].value_counts().sort_index()
plt.figure(figsize=(10,5))
sns.lineplot(x=yearly_counts.index, y=yearly_counts.values)
plt.title("Crimes per Year in Chicago (2019–Present)")
plt.xlabel("Year")
plt.ylabel("Number of Crimes")
plt.show()

# Crimes per month
df['month'] = pd.to_datetime(df['date']).dt.month
monthly_counts = df['month'].value_counts().sort_index()
sns.barplot(x=monthly_counts.index, y=monthly_counts.values)
plt.title("Crimes by Month")
plt.xlabel("Month")
plt.ylabel("Number of Crimes")
plt.show()


Top Crime Types

In [ ]:
plt.figure(figsize=(10,6))
df['primary_type'].value_counts().head(10).plot(kind='bar')
plt.title("Top 10 Crime Types in Chicago (2019–Present)")
plt.xlabel("Crime Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()


Crimes by Location Type

In [ ]:
plt.figure(figsize=(10,6))
df['location_description'].value_counts().head(10).plot(kind='barh')
plt.title("Top 10 Locations of Crime Occurrence")
plt.xlabel("Number of Crimes")
plt.ylabel("Location Type")
plt.show()




In [ ]:
sns.kdeplot(
    data=df.sample(10000),
    x='longitude',
    y='latitude',
    fill=True,
    cmap='Reds',
    alpha=0.6
)
plt.title("Crime Density Across Chicago")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


Arrest and Domestic Patterns

In [ ]:
# Arrest ratio
plt.figure(figsize=(5,5))
df['arrest'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title("Arrest vs Non-Arrest Cases")
plt.ylabel("")
plt.show()

# Domestic incidents
plt.figure(figsize=(5,5))
df['domestic'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title("Domestic vs Non-Domestic Crimes")
plt.ylabel("")
plt.show()


Crimes by Community Area

In [ ]:
plt.figure(figsize=(12,6))
df['community_area'].value_counts().head(15).plot(kind='bar')
plt.title("Top 15 Community Areas by Crime Count")
plt.xlabel("Community Area")
plt.ylabel("Number of Crimes")
plt.show()


Temporal Trends by Day and Hour

In [ ]:
df['day_of_week'] = pd.to_datetime(df['date']).dt.day_name()
df['hour'] = pd.to_datetime(df['date']).dt.hour

# Day of week
plt.figure(figsize=(8,5))
sns.countplot(x='day_of_week', data=df, order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
plt.title("Crimes by Day of the Week")
plt.xticks(rotation=45)
plt.show()

# Hour of day
plt.figure(figsize=(10,5))
sns.countplot(x='hour', data=df)
plt.title("Crimes by Hour of the Day")
plt.xlabel("Hour")
plt.ylabel("Count")
plt.show()


Correlation Heatmap (for Numeric Columns)

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Between Numerical Features")
plt.show()


Geospatial Visualization

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='longitude', y='latitude', data=df.sample(10000), alpha=0.3)
plt.title("Crime Locations in Chicago")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


summary

Crimes peaked in 2022 and are slightly declining after that.

Most frequent crimes: Theft, Battery, and Criminal Damage.

Hotspots: South and West Chicago areas.

Crimes are most common in summer and weekends, especially at night.

Arrests occur in roughly X% of cases.